# Crop Disease & Pest Classifier — Kaggle GPU Pipeline

Run the complete pipeline (download -> clean -> pHash leak-free split -> 2-stage training -> calibration/OOD eval -> ONNX export & CPU benchmark) on a free Kaggle GPU (**T4 x2** or **P100**).

**Before running:**
1. In Kaggle right sidebar -> **Session options -> Accelerator**: select **GPU T4 x2** or **GPU P100**.
2. Toggle **Internet** -> **On** (required to download PlantVillage, PlantDoc, and pretrained `timm` weights).

In [ ]:
# 1. Verify Kaggle GPU
!nvidia-smi
import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()} | Count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| Capability:", torch.cuda.get_device_capability(0))

In [ ]:
# 2. Navigate into cloned repository
import os

# If cloned into 'crop-disease-classifier' or 'crop-disease', navigate to whichever exists
for path in ["/kaggle/working/crop-disease-classifier", "/kaggle/working/crop-disease"]:
    if os.path.exists(os.path.join(path, "ml")):
        WORK_DIR = path
        break
else:
    WORK_DIR = "/kaggle/working/crop-disease-classifier"
    !git clone https://github.com/PRAKHAR-MOHAN/crop-disease-classifier.git {WORK_DIR}

%cd {WORK_DIR}
%env PYTHONPATH={WORK_DIR}
!ls -la


In [ ]:
# 3. Install only missing/extra packages (keeps Kaggle's preinstalled CUDA PyTorch intact)
!pip install -q ImageHash onnx onnxruntime onnxscript "timm>=1.0.9" "albumentations>=1.4.14" optuna

In [ ]:
# 4. Download PlantVillage, PlantDoc (real-field), and CIFAR-10 non-plant OOD negatives
!python -m ml.data.download --datasets plantvillage plantdoc ood_non_plant

In [ ]:
# 5. Clean, deduplicate (SHA-256), standardize to Crop___Disease, and run leak-free pHash 70/15/15 split
# --materialize writes fast <=512px JPEGs for faster epoch I/O on Kaggle disks
!python -m ml.data.clean_and_merge --materialize
!python -m ml.data.split
!python -m ml.data.dataset_card

In [ ]:
# 6A. (Fast) Train EfficientNetV2-S with 2-stage transfer learning (~10-15 min on Kaggle GPU)
!python -m ml.training.train --config configs/train_efficientnetv2_s.yaml

# 6B. (Optional) Compare all 3 architectures (EfficientNetV2-S vs ConvNeXt-Tiny vs Swin-Tiny)
# !python -m ml.training.train --compare configs/train_efficientnetv2_s.yaml configs/train_convnext_tiny.yaml configs/train_swin_tiny.yaml

In [ ]:
# 7. Train lightweight Plant/Pest vs. Non-Plant OOD gate
!python -m ml.models.plant_gate --epochs 4

In [ ]:
# 8. Evaluate In-Domain Test vs. PlantDoc Real-Field Gap, Temperature Scaling & OOD Thresholds
!python -m ml.evaluation.evaluate --checkpoint artifacts/best_model.pt

In [ ]:
# 9. Export dual-output ONNX (logits + Grad-CAM), TorchScript, and benchmark CPU latency
!python -m ml.export.export_model --checkpoint artifacts/best_model.pt --torchscript --quantize-int8
!python -m ml.export.benchmark_cpu --onnx artifacts/model.onnx --checkpoint artifacts/best_model.pt

In [ ]:
# 10. Package trained artifacts & reports for easy download from Kaggle Output panel
!zip -r /kaggle/working/crop_disease_outputs.zip artifacts/ reports/
!ls -lh /kaggle/working/crop_disease_outputs.zip